# 06 · Luật chuyển trạng thái và các bộ điều kiện hoá

Classifier dự đoán từng đoạn độc lập. Nhưng `fallen` (nằm sau té) gần như luôn đến sau `fall`. Các bộ điều kiện hoá dùng **quá khứ** của chuỗi dự đoán để sửa dự đoán hiện tại:

| Bộ | Ý tưởng | Học tham số? |
|---|---|---|
| `rule_condition` | Thấy `fall` → kéo các đoạn sau về `fallen`, trọng số giảm dần theo thời gian | Không |
| `hmm_filter` | Lọc tiến HMM với ma trận chuyển ước lượng từ nhãn train | Không (đếm) |
| `prior_shift` | Sửa lệch phân bố lớp giữa miền nguồn và miền đích | Không |
| `LearnableConditioner` (Hybrid) | Self-attention nhân quả trên chuỗi logits, học phần điều chỉnh | Có |

Tất cả đều **nhân quả** (chỉ dùng quá khứ) nên dùng được khi chạy thời gian thực.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# Notebook nằm trong thư mục notebooks/ -> thêm thư mục gốc repo vào sys.path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

Repo root: <repo>


**`tcfall/conditioners.py` · phần import và hằng số**

Import; `TRANSIENT_TO_STATIC` cho biết hành động chuyển tiếp nào dẫn tới trạng thái tĩnh nào.

In [2]:
from __future__ import annotations
import sys
import numpy as np
import torch
from tcfall.config import CKPT_DIR, NUM_CLASSES, ROOT, TRANSIENT_TO_STATIC
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.conditioning import LearnableConditioner  # noqa: E402

**`tcfall/conditioners.py` · rule_condition**

**Luật chuyển trạng thái** (không có tham số học):
- Duyệt chuỗi theo thời gian. Khi đoạn `t` được dự đoán là hành động chuyển tiếp (ví dụ `fall`), ghi nhớ trạng thái tĩnh tương ứng (`fallen`) cùng thời điểm.
- Với các đoạn sau **không** phải hành động chuyển tiếp: trộn xác suất với vector one-hot của trạng thái tĩnh, trọng số `w = 0.5^(Δt/τ)` (giảm một nửa sau mỗi `τ` giây); quá cửa sổ `W` giây thì quên.
- Hai sửa lỗi quan trọng so với bản đầu: không boost chính đoạn `fall` (nếu không mọi `fall` biến thành `fallen`), và không boost đoạn `fall` thứ hai liên tiếp.

In [3]:
def rule_condition(probs: np.ndarray, starts: np.ndarray, tau: float = 4.0, window: float = 10.0) -> np.ndarray:
    """probs (T, C) -> probs đã điều kiện hoá. Giống hệt RuleBasedConditioner (đã sửa lỗi):
    không boost segment đang được dự đoán là 1 hành động chuyển tiếp; Δt=0 không được boost."""
    out = probs.copy()
    last: dict[int, float] = {}
    pred = probs.argmax(1)
    for t in range(len(probs)):
        if pred[t] not in TRANSIENT_TO_STATIC:
            for s, t0 in list(last.items()):
                dt = starts[t] - t0
                if dt > window:
                    del last[s]
                    continue
                w = 0.5 ** (dt / tau)
                prior = np.zeros(probs.shape[1]); prior[s] = 1.0
                out[t] = (1 - w) * out[t] + w * prior
        else:
            last[TRANSIENT_TO_STATIC[int(pred[t])]] = starts[t]
    return out / out.sum(1, keepdims=True).clip(min=1e-8)

**Ví dụ nhỏ:** chuỗi 6 đoạn, mô hình chỉ thấy `fall` ở đoạn 2 và phân vân giữa `fallen`/`lying` ở các đoạn sau.

In [4]:
import numpy as np
from tcfall.config import CLASS_NAMES
P = np.full((6, 10), 0.02)
P[0, 0] = P[1, 0] = 0.8                     # walk
P[2, 1] = 0.8                               # fall
for t in (3, 4, 5):
    P[t, 6] = 0.45; P[t, 2] = 0.37          # lying nhỉnh hơn fallen
P /= P.sum(1, keepdims=True)
starts = np.arange(6) * 1.5
print("trước:", [CLASS_NAMES[k] for k in P.argmax(1)])
print("sau  :", [CLASS_NAMES[k] for k in rule_condition(P, starts, tau=4, window=10).argmax(1)])

trước: ['walk', 'walk', 'fall', 'lying', 'lying', 'lying']
sau  : ['walk', 'walk', 'fall', 'fallen', 'fallen', 'fallen']


**`tcfall/conditioners.py` · estimate_transitions**

Ma trận chuyển HMM từ ma trận đếm giữa các segment: gán xác suất giữ nguyên lớp `self_loop`, phần còn lại chia theo tần suất chuyển tiếp (có làm trơn +1).

In [5]:
def estimate_transitions(counts: np.ndarray, self_loop: float, smoothing: float = 1.0) -> np.ndarray:
    """Ma trận chuyển tiếp HMM từ ma trận đếm chuyển nhãn giữa 2 segment liên tiếp.
    self_loop: xác suất giữ nguyên lớp (segment liên tiếp trong nhãn gốc luôn khác lớp,
    nhưng dự đoán thì có thể lặp lại), phần còn lại chia theo tần suất chuyển tiếp."""
    off = counts.astype(float) + smoothing
    np.fill_diagonal(off, 0.0)
    off = off / off.sum(1, keepdims=True)
    return self_loop * np.eye(NUM_CLASSES) + (1 - self_loop) * off

**`tcfall/conditioners.py` · transitions_from_counts**

Chuẩn hoá hàng của ma trận đếm lấy mẫu theo thời gian (giữ đường chéo). Đây là cách tạo ma trận `A` dùng trong kết quả cuối.

In [6]:
def transitions_from_counts(counts: np.ndarray, smoothing: float = 1.0) -> np.ndarray:
    """Ma trận chuyển tiếp chuẩn hoá theo hàng, GIỮ đường chéo (dùng cho đếm lấy mẫu theo thời gian)."""
    M = counts.astype(float) + smoothing
    return M / M.sum(1, keepdims=True)

**`tcfall/conditioners.py` · sharpen_self_loop**

Điều chỉnh độ "dính" của trạng thái: `γ > 1` tăng xác suất ở lại, `γ < 1` giảm; là một tham số được dò trên val.

In [7]:
def sharpen_self_loop(A: np.ndarray, gamma: float) -> np.ndarray:
    """Điều chỉnh độ 'dính' của trạng thái: gamma>1 tăng self-loop, <1 giảm, rồi chuẩn hoá lại."""
    B = A.copy()
    np.fill_diagonal(B, np.diag(A) ** (1.0 / gamma))
    return B / B.sum(1, keepdims=True)

**`tcfall/conditioners.py` · hmm_filter**

**Lọc tiến HMM** (forward filtering): `α_t ∝ (Aᵀ α_{t−1}) ⊙ (p_t / prior)^strength`.
- `p_t / prior` đổi xác suất hậu nghiệm của classifier thành *likelihood* (chia cho prior lớp của dữ liệu train).
- `strength` điều chỉnh mức tin vào classifier so với ma trận chuyển.
- Chỉ dùng quá khứ, chi phí `O(T·C²)`.

In [8]:
def hmm_filter(probs: np.ndarray, A: np.ndarray, class_prior: np.ndarray, strength: float = 1.0) -> np.ndarray:
    """Lọc tiến: alpha_t ∝ (A^T alpha_{t-1}) * (p_t / prior)^strength. Nhân quả, O(T·C²)."""
    lik = (probs / class_prior[None]) ** strength
    alpha = np.zeros_like(probs)
    a = np.full(probs.shape[1], 1.0 / probs.shape[1])
    for t in range(len(probs)):
        pred = a @ A if t > 0 else a
        a = pred * lik[t]
        a = a / a.sum().clip(min=1e-12)
        alpha[t] = a
    return alpha

**`tcfall/conditioners.py` · prior_shift**

Sửa lệch phân bố lớp (Saerens et al., 2002): nhân xác suất với `(π_đích / π_nguồn)^α` rồi chuẩn hoá lại.

In [9]:
def prior_shift(probs: np.ndarray, source_prior: np.ndarray, target_prior: np.ndarray, alpha: float = 1.0) -> np.ndarray:
    """Hiệu chỉnh label shift (Saerens et al., 2002): p'(y|x) ∝ p(y|x) · (π_t / π_s)^alpha."""
    w = (target_prior / source_prior) ** alpha
    q = probs * w[None]
    return q / q.sum(1, keepdims=True)

**`tcfall/conditioners.py` · em_target_prior**

Ước lượng phân bố lớp của miền đích **không cần nhãn** bằng EM: lặp sửa xác suất theo prior hiện tại rồi lấy trung bình làm prior mới.

In [10]:
def em_target_prior(probs: np.ndarray, source_prior: np.ndarray, iters: int = 100, tol: float = 1e-6) -> np.ndarray:
    """Ước lượng prior miền đích KHÔNG cần nhãn bằng EM (Saerens et al., 2002)."""
    pi = source_prior.copy()
    for _ in range(iters):
        q = prior_shift(probs, source_prior, pi)
        new = q.mean(0)
        if np.abs(new - pi).max() < tol:
            pi = new
            break
        pi = new
    return pi

## Hybrid: `LearnableConditioner` (`src/conditioning.py`)

Bộ điều kiện hoá học được: self-attention trên chuỗi logits sau luật, **chỉ nhìn quá khứ trong 15 giây**, và chỉ học **phần điều chỉnh** cộng vào logits gốc.

**`src/conditioning.py` · phần import và hằng số**

Import và ánh xạ lớp từ `src/data.py`.

In [11]:
from __future__ import annotations
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.data import TRANSIENT_TO_STATIC, CLASS_TO_ID

**`src/conditioning.py` · SinusoidalTimeEncoding**

Positional encoding theo **thời gian thật (giây)** thay vì chỉ số thứ tự, để mô hình biết khoảng cách thời gian thực giữa các đoạn (video có nhịp khác nhau).

In [12]:
class SinusoidalTimeEncoding(nn.Module):
    """Positional encoding dựa trên thời gian thực (giây), thay vì chỉ số thứ tự segment,
    để model biết được khoảng cách thời gian thực giữa các segment (video có tốc độ khác nhau)."""

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, start_times: torch.Tensor) -> torch.Tensor:
        times = start_times.float()  # (batch, seq_len), đơn vị giây
        device = start_times.device
        div_term = torch.exp(
            torch.arange(0, self.dim, 2, device=device).float() * (-math.log(10000.0) / self.dim)
        )
        pe = torch.zeros(*times.shape, self.dim, device=device)
        pe[..., 0::2] = torch.sin(times.unsqueeze(-1) * div_term)
        pe[..., 1::2] = torch.cos(times.unsqueeze(-1) * div_term)
        return pe

**`src/conditioning.py` · class LearnableConditioner**

Khai báo các lớp con:
- `input_proj`: chiếu logits 10 chiều lên 128 chiều; cộng encoding thời gian.
- 1 khối Transformer: `MultiheadAttention` (4 head) → `LayerNorm` → FFN (GELU) → `LayerNorm`.
- `classifier`: ra 10 lớp. Với `residual=True`, khởi tạo bằng 0 nên lúc đầu đầu ra **đúng bằng** logits gốc — mô hình không tệ hơn baseline ngay từ đầu rồi học dần phần sửa.

In [13]:
class LearnableConditioner(nn.Module):
    """
    Bản đầy đủ: dùng self-attention theo thời gian để model tự học cách kết hợp
    thông tin từ các segment trước đó (thay vì rule cứng). Đây là phần đóng góp
    kỹ thuật chính của dự án — thay thế RuleBasedConditioner sau khi pilot xác nhận
    hypothesis có tín hiệu tích cực.
    """

    def __init__(self, feature_dim: int, num_classes: int = 16,
                 hidden_dim: int = 128, num_heads: int = 4,
                 context_window_seconds: float = 15.0, residual: bool = False,
                 dropout: float = 0.0):
        """
        residual: khi True, input được coi là LOGITS của 1 classifier khác đã train sẵn
            (vd baseline per-segment, không có context chéo segment) — feature_dim PHẢI
            bằng num_classes. Module này khi đó chỉ học phần ĐIỀU CHỈNH (delta) cộng vào
            logits gốc, thay vì học lại phân loại từ đầu trên representation nghèo hơn.
            Init classifier gần 0 để lúc bắt đầu train, output ~ logits gốc (không tệ hơn
            baseline), rồi học dần cách dùng ngữ cảnh thời gian để chỉnh cho tốt hơn.
        """
        super().__init__()
        self.context_window_seconds = context_window_seconds
        self.residual = residual
        if residual and feature_dim != num_classes:
            raise ValueError("residual=True yêu cầu feature_dim == num_classes (input phải là logits).")

        self.input_proj = nn.Linear(feature_dim, hidden_dim)
        self.pos_encoding = SinusoidalTimeEncoding(hidden_dim)

        self.self_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, batch_first=True, dropout=dropout
        )
        self.attn_dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.classifier = nn.Linear(hidden_dim, num_classes)
        if residual:
            # Khởi tạo gần 0 để delta ban đầu ~ 0 -> output ban đầu ~ logits gốc của baseline.
            nn.init.zeros_(self.classifier.weight)
            nn.init.zeros_(self.classifier.bias)

**`src/conditioning.py` · LearnableConditioner._build_time_mask**

Mặt nạ attention **nhân quả + cửa sổ**: đoạn `i` chỉ được nhìn đoạn `j` nếu `0 ≤ t_i − t_j ≤ 15 s` và `j` không phải padding. Đây là ràng buộc của hệ thống thật: không biết tương lai.

In [14]:
def _build_time_mask(self, start_times: torch.Tensor,
                      key_padding_mask: torch.Tensor) -> torch.Tensor:
    """
    Chỉ cho phép attention nhìn về QUÁ KHỨ trong phạm vi context_window_seconds
    (causal + windowed attention) — mô phỏng đúng ràng buộc thực tế: hệ thống
    chỉ biết được các sự kiện ĐÃ xảy ra, không biết tương lai.
    """
    batch_size, seq_len = start_times.shape
    times = start_times.float()  # (batch, seq_len), đơn vị giây

    time_diff = times.unsqueeze(2) - times.unsqueeze(1)  # (batch, seq_len, seq_len): query - key
    causal = time_diff >= 0  # chỉ nhìn về quá khứ hoặc hiện tại
    within_window = time_diff <= self.context_window_seconds
    allowed = causal & within_window

    # Kết hợp với padding mask (loại bỏ segment padding)
    pad_allowed = key_padding_mask.unsqueeze(1).expand(-1, seq_len, -1)
    allowed = allowed & pad_allowed

    # torch.nn.MultiheadAttention cần attn_mask dạng additive (True = bị chặn -> -inf)
    attn_mask = ~allowed
    # Mở rộng cho multi-head: (batch * num_heads, seq_len, seq_len) sẽ được xử lý ở forward
    return attn_mask


LearnableConditioner._build_time_mask = _build_time_mask  # gắn method vào class đã khai báo ở cell trước

**`src/conditioning.py` · LearnableConditioner.forward**

Chiếu + encoding thời gian → self-attention có mặt nạ → FFN → classifier; với `residual`, cộng logits gốc vào đầu ra.

In [15]:
def forward(self, features: torch.Tensor, start_times: torch.Tensor,
            mask: torch.Tensor) -> torch.Tensor:
    """
    features: (batch, seq_len, feature_dim)
    start_times: (batch, seq_len) — thời điểm bắt đầu mỗi segment, ĐƠN VỊ GIÂY
    mask: (batch, seq_len) — True = segment hợp lệ

    Trả về logits: (batch, seq_len, num_classes)
    """
    batch_size, seq_len, _ = features.shape

    x = self.input_proj(features)
    x = x + self.pos_encoding(start_times)

    attn_mask = self._build_time_mask(start_times, mask)  # (batch, seq_len, seq_len)
    # Lặp lại cho từng head vì nn.MultiheadAttention với batch_first cần (batch*heads, L, L)
    num_heads = self.self_attn.num_heads
    attn_mask_expanded = attn_mask.repeat_interleave(num_heads, dim=0)

    key_padding_mask = ~mask  # True = bị bỏ qua

    attn_out, _ = self.self_attn(
        x, x, x,
        attn_mask=attn_mask_expanded,
        key_padding_mask=key_padding_mask,
        need_weights=False,
    )
    x = self.norm1(x + self.attn_dropout(attn_out))
    x = self.norm2(x + self.ffn(x))

    logits = self.classifier(x)
    if self.residual:
        logits = logits + features  # học phần điều chỉnh (delta) cộng lên logits gốc
    return logits


LearnableConditioner.forward = forward  # gắn method vào class đã khai báo ở cell trước

**Ví dụ nhỏ:** chạy Hybrid chưa huấn luyện; vì `classifier` khởi tạo 0 nên đầu ra trùng logits đầu vào.

In [16]:
import torch
m = LearnableConditioner(feature_dim=10, num_classes=10, residual=True).eval()
x = torch.randn(1, 6, 10); t = torch.arange(6.0)[None] * 1.5; mask = torch.ones(1, 6, dtype=torch.bool)
with torch.no_grad():
    print(torch.allclose(m(x, t, mask), x))   # True

True


**`tcfall/conditioners.py` · load_hybrid**

Dựng `LearnableConditioner` với cấu hình đã huấn luyện (128 chiều, 4 head, cửa sổ 15 s, residual) và nạp checkpoint.

In [17]:
def load_hybrid(path, device: str = "cpu") -> LearnableConditioner:
    m = LearnableConditioner(feature_dim=NUM_CLASSES, num_classes=NUM_CLASSES, hidden_dim=128, num_heads=4,
                             context_window_seconds=15.0, residual=True, dropout=0.2)
    m.load_state_dict(torch.load(str(path), map_location=device, weights_only=True))
    return m.to(device).eval()

**`tcfall/conditioners.py` · hybrid_apply**

Áp Hybrid lên chuỗi xác suất sau luật: lấy log, chạy model, softmax lại.

In [18]:
@torch.no_grad()
def hybrid_apply(model: LearnableConditioner, rule_probs: np.ndarray, starts: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    dev = next(model.parameters()).device
    x = torch.tensor(np.log(rule_probs + eps), dtype=torch.float32, device=dev)[None]
    s = torch.tensor(starts, dtype=torch.float32, device=dev)[None]
    m = torch.ones(1, len(starts), dtype=torch.bool, device=dev)
    return torch.softmax(model(x, s, m), -1)[0].cpu().numpy()

## ▶ Chạy thật trên dữ liệu thật

Phần trên định nghĩa code. Từ đây là các ô chạy thật bằng dữ liệu, checkpoint và GPU của dự án.

Áp từng cách làm mượt lên xác suất thật của tập val, cùng một phép đo.

In [19]:
import json, pickle
import numpy as np, pandas as pd, torch
from tcfall.config import CKPT_DIR, CLASS_NAMES, OUTPUTS
from tcfall.data import clean_labels, load_labels, load_split, transition_counts_timesampled
from tcfall.metrics import core_metrics

cache = pickle.load(open(OUTPUTS / "logits" / "timefix_val.pkl", "rb"))
videos = {p: (torch.softmax(torch.tensor(v["crops"]).float(), -1).mean(1).numpy(), v["starts"], v["labels"]) for p, v in cache.items()}
best = json.loads((OUTPUTS / "best_config.json").read_text(encoding="utf-8"))["stream_ft"]
datasets = ("of-syn", "gmdcsa24", "le2i", "caucafall")
source = pd.concat([clean_labels(load_labels(d))[0] for d in datasets])
sp = {}
for d in datasets:
    sp.update(load_split(d))
A = transitions_from_counts(transition_counts_timesampled(source[source.path.map(sp) == "train"], 0.5))
prior = np.array(best["hmm_prior"])
hybrid = load_hybrid(CKPT_DIR / "learnable_conditioner_v8_stsyn_kl0.3_seed42_best.pt")
methods = {"mô hình (không làm mượt)": lambda P, s: P,
           "quy tắc τ=1, W=5": lambda P, s: rule_condition(P, s, 1.0, 5.0),
           "quy tắc τ=4, W=10": lambda P, s: rule_condition(P, s, 4.0, 10.0),
           "HMM": lambda P, s: hmm_filter(P, A, prior, 1.0),
           "Hybrid (attention nhỏ)": lambda P, s: hybrid_apply(hybrid, rule_condition(P, s, 4.0, 10.0), s)}
y = np.concatenate([v[2] for v in videos.values()])
rows = [{"cách": n, **{k: round(x, 2) for k, x in core_metrics(y, np.concatenate([f(P, s).argmax(1) for P, s, _ in videos.values()])).items()}} for n, f in methods.items()]
print(pd.DataFrame(rows).to_string(index=False))

                    cách  fallen_f1  fall_f1  union_f1  union_se  union_sp  bacc  macro_f1
mô hình (không làm mượt)      26.87    41.53     58.40     52.14     86.25 25.53     23.81
        quy tắc τ=1, W=5      27.50    41.53     58.56     55.00     82.90 23.86     22.00
       quy tắc τ=4, W=10      29.17    41.53     58.78     58.57     78.81 23.90     21.91
                     HMM      20.29    18.71     32.69     24.29     87.36 17.89     17.08
  Hybrid (attention nhỏ)      33.33    43.68     58.52     56.43     81.04 25.54     25.96


Nhìn công thức làm việc trên một video thật: đoạn nào được đổi nhãn và độ tin thay đổi ra sao.

In [20]:
path = next(p for p, (P, s, l) in videos.items() if (P.argmax(1) != rule_condition(P, s, 1.0, 5.0).argmax(1)).any())
P, s, l = videos[path]
R = rule_condition(P, s, 1.0, 5.0)
print(pd.DataFrame({"bắt đầu (s)": s.round(2), "nhãn thật": [CLASS_NAMES[i] for i in l], "mô hình": [CLASS_NAMES[i] for i in P.argmax(1)],
                    "sau quy tắc": [CLASS_NAMES[i] for i in R.argmax(1)], "P(đã ngã) trước": P[:, 2].round(3), "P(đã ngã) sau": R[:, 2].round(3)}).to_string(index=False))

 bắt đầu (s) nhãn thật mô hình sau quy tắc  P(đã ngã) trước  P(đã ngã) sau
        0.00     other   other       other            0.041          0.041
        3.83      fall    fall        fall            0.110          0.110
        5.58      walk   other      fallen            0.121          0.381
        7.29      fall    walk        walk            0.098          0.180
        9.24  stand_up    walk        walk            0.085          0.085
       10.02      walk    walk        walk            0.052          0.052
